# WM-811K 견고성 증강·2단계 분류 후보 실험

기존 배포 모델을 덮어쓰지 않고 다음 세 후보를 동일한 lot-aware validation에서 비교합니다.

- baseline: 현재와 같은 9-class flat CNN
- robust_flat: 4px 이동과 활성 die 누락을 학습에 추가한 flat CNN
- robust_hierarchical: 정상/불량 판별 후 불량 유형을 분류하는 2단계 CNN

seed 17, 42, 2026을 모두 실행합니다. 후보 선택 중에는 **test 추론은 수행하지 않습니다**. 실행 결과가 기준을 통과해도 기존 모델은 자동 교체되지 않습니다.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import json, shutil, subprocess, sys
import pandas as pd
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형에서 T4 GPU를 선택한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))

DATA_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
MAPS = DATA_DIR / 'wafer_maps_64.npy'
LABELS = DATA_DIR / 'labels.npy'
for path in (MAPS, LABELS):
    assert path.is_file(), f'전처리 파일이 없습니다: {path}'
OUTPUT_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_robust_hierarchical/validation_multiseed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('결과 저장:', OUTPUT_DIR)


## 로컬 파일 3개를 차례로 선택

파일 이름 뒤에 `(1)`이 붙어도 셀에서 정식 이름으로 저장하므로 괜찮습니다.

1. `코드/wm811k/train_wm811k_cnn.py`
2. `코드/wm811k/train_wm811k_robust_hierarchical_candidates.py`
3. `결과물/wm811k/split_results/split_assignments.csv`


In [ ]:
def upload_as(target_name):
    print(f'{target_name} 파일을 선택하세요.')
    uploaded = files.upload()
    assert len(uploaded) == 1, '이번 선택창에서는 파일 하나만 선택하세요.'
    _, content = next(iter(uploaded.items()))
    target = Path('/content') / target_name
    target.write_bytes(content)
    print('저장 완료:', target, target.stat().st_size, 'bytes')
    return target

BASE_SCRIPT = upload_as('train_wm811k_cnn.py')
EXPERIMENT_SCRIPT = upload_as('train_wm811k_robust_hierarchical_candidates.py')
ASSIGNMENTS = upload_as('split_assignments.csv')


## T4 GPU 학습

세 후보를 3개 seed로 검증하므로 기존 단일 학습보다 오래 걸립니다. 중단 후 같은 셀을 다시 실행하면 완료된 stage는 재사용합니다.


In [ ]:
command = [
    sys.executable, str(EXPERIMENT_SCRIPT),
    '--maps', str(MAPS),
    '--labels', str(LABELS),
    '--assignments', str(ASSIGNMENTS),
    '--output-dir', str(OUTPUT_DIR),
    '--seeds', '17', '42', '2026',
    '--epochs', '20',
    '--batch-size', '256',
    '--num-workers', '2',
]
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
assert return_code == 0, f'학습 스크립트가 종료 코드 {return_code}로 실패했습니다.'
print('검증 학습 완료:', OUTPUT_DIR)


In [ ]:
selection = json.loads((OUTPUT_DIR / 'selection_summary.json').read_text(encoding='utf-8'))
display(pd.read_csv(OUTPUT_DIR / 'seed_candidate_metrics.csv'))
print(json.dumps(selection, ensure_ascii=False, indent=2))
if selection['selected_candidate'] == 'baseline':
    print('결론: 기존 모델 유지. 배포 모델은 변경하지 않습니다.')
else:
    print('결론: validation 관문 통과 후보 =', selection['selected_candidate'])
    print('아직 배포하지 말고 결과 ZIP을 로컬 프로젝트에 전달하세요.')


## 전달용 결과 ZIP 다운로드

요약 결과와 선택 후보의 seed 42 체크포인트만 포함합니다. 전체 중간 결과는 Google Drive에 그대로 남습니다.


In [ ]:
EXPORT_DIR = Path('/content/wm811k_robust_hierarchical_results')
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir()
for name in ('protocol.json', 'selection_summary.json', 'seed_candidate_metrics.csv', 'metric_summary.csv', 'candidate_comparison.png', 'README.md'):
    shutil.copy2(OUTPUT_DIR / name, EXPORT_DIR / name)
selected = selection['selected_candidate']
selected_source = OUTPUT_DIR / 'runs' / 'seed_42' / selected
selected_target = EXPORT_DIR / 'selected_seed_42' / selected
selected_target.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(selected_source, selected_target)
zip_base = Path('/content/wm811k_robust_hierarchical_results')
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=EXPORT_DIR))
print('ZIP:', zip_path, f'{zip_path.stat().st_size / 1024**2:.1f} MB')
files.download(str(zip_path))
